In [21]:
import pandas as pd  # Importing pandas for data handling
import numpy as np  # Importing numpy for numerical operations
import tensorflow as tf  # Importing TensorFlow for neural network implementation
from tensorflow import keras  # Importing Keras, a high-level API for TensorFlow
from sklearn.model_selection import train_test_split  # Importing train_test_split for data splitting
from sklearn.preprocessing import StandardScaler  # Importing StandardScaler for feature scaling
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score  # Importing evaluation metrics

# Step 1: Load the dataset
df = pd.read_csv("/content/Alphabets_data.csv")  # Reading the dataset

# Display first few rows of the dataset
display(df.head())

,letter,xbox,ybox,width,height,onpix,xbar,ybar,x2bar,y2bar,xybar,x2ybar,xy2bar,xedge,xedgey,yedge,yedgex
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


In [23]:
# Step 2: Data Exploration
print("Dataset Shape:", df.shape)  # Checking dataset dimensions
print("Columns:", df.columns)  # Listing column names
print("Missing Values:", df.isnull().sum())  # Checking for missing values
# Check if 'label' column exists, if not, try 'Letter' based on global variable preview
if 'label' in df.columns:
    print("Unique Classes:", df['label'].nunique())  # Checking number of unique labels using 'label' column if it exists
elif 'letter' in df.columns:
    print("Unique Classes:", df['letter'].nunique())  # Checking number of unique labels using 'letter' column if it exists and 'label' does not
else:
    print("Error: Neither 'label' nor 'letter' column found in the DataFrame.")

Dataset Shape: (20000, 17)
Columns: Index(['letter', 'xbox', 'ybox', 'width', 'height', 'onpix', 'xbar', 'ybar',
       'x2bar', 'y2bar', 'xybar', 'x2ybar', 'xy2bar', 'xedge', 'xedgey',
       'yedge', 'yedgex'],
      dtype='object')
Missing Values: letter    0
xbox      0
ybox      0
width     0
height    0
onpix     0
xbar      0
ybar      0
x2bar     0
y2bar     0
xybar     0
x2ybar    0
xy2bar    0
xedge     0
xedgey    0
yedge     0
yedgex    0
dtype: int64
Unique Classes: 26


In [30]:
# Step 3: Data Preprocessing
X = df.drop(columns=['letter'])  # Extracting features
y = df['letter']  # Extracting target variable

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Encode target variable
y_encoded = pd.get_dummies(y).values  # One-hot encoding labels

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)


In [31]:
# Step 4: Define the Neural Network Model
def build_model(hidden_layers=1, neurons=64, activation='relu', learning_rate=0.001):
    model = keras.Sequential()  # Initializing a sequential model
    model.add(keras.layers.Input(shape=(X_train.shape[1],)))  # Input layer

    # Adding hidden layers
    for _ in range(hidden_layers):
        model.add(keras.layers.Dense(neurons, activation=activation))

    model.add(keras.layers.Dense(y_train.shape[1], activation='softmax'))  # Output layer

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)  # Optimizer
    model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])  # Compile model
    return model


In [32]:
# Step 5: Train the Model
model = build_model()
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))


Epoch 1/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.3071 - loss: 2.5978 - val_accuracy: 0.6948 - val_loss: 1.2333
Epoch 2/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7062 - loss: 1.1225 - val_accuracy: 0.7732 - val_loss: 0.8740
Epoch 3/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7728 - loss: 0.8376 - val_accuracy: 0.8002 - val_loss: 0.7252
Epoch 4/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7998 - loss: 0.7135 - val_accuracy: 0.8220 - val_loss: 0.6393
Epoch 5/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8280 - loss: 0.6150 - val_accuracy: 0.8413 - val_loss: 0.5764
Epoch 6/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8466 - loss: 0.5477 - val_accuracy: 0.8482 - val_loss: 0.5317
Epoch 7/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8611 - loss: 0.4951 - val_accuracy: 0.8525 - val_loss: 0.4986
Epoch 8/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8668 - loss: 0.4721 - val_accuracy: 0.

In [33]:
# Step 6: Evaluate the Model
y_pred_prob = model.predict(X_test)  # Get predictions
y_pred = np.argmax(y_pred_prob, axis=1)  # Convert probabilities to class labels
y_test_actual = np.argmax(y_test, axis=1)  # Convert one-hot labels back to original

# Calculate evaluation metrics
accuracy = accuracy_score(y_test_actual, y_pred)
precision = precision_score(y_test_actual, y_pred, average='weighted')
recall = recall_score(y_test_actual, y_pred, average='weighted')
f1 = f1_score(y_test_actual, y_pred, average='weighted')

# Display results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Accuracy: 0.9147
Precision: 0.9163
Recall: 0.9147
F1 Score: 0.9149


In [34]:
# Step 7: Hyperparameter Tuning (Example with Grid Search)
best_model = None
best_accuracy = 0

for hl in [1, 2]:  # Varying number of hidden layers
    for neurons in [32, 64]:  # Varying number of neurons
        for lr in [0.001, 0.01]:  # Varying learning rate
            print(f"Training with {hl} hidden layers, {neurons} neurons per layer, learning rate {lr}")
            temp_model = build_model(hidden_layers=hl, neurons=neurons, learning_rate=lr)
            temp_model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
            temp_pred_prob = temp_model.predict(X_test)
            temp_pred = np.argmax(temp_pred_prob, axis=1)
            temp_accuracy = accuracy_score(y_test_actual, temp_pred)
            print(f"Validation Accuracy: {temp_accuracy:.4f}")

            if temp_accuracy > best_accuracy:
                best_model = temp_model
                best_accuracy = temp_accuracy

print("Best Model Accuracy:", best_accuracy)


Training with 1 hidden layers, 32 neurons per layer, learning rate 0.001
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Validation Accuracy: 0.8460
Training with 1 hidden layers, 32 neurons per layer, learning rate 0.01
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Validation Accuracy: 0.8925
Training with 1 hidden layers, 64 neurons per layer, learning rate 0.001
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Validation Accuracy: 0.8832
Training with 1 hidden layers, 64 neurons per layer, learning rate 0.01
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Validation Accuracy: 0.9175
Training with 2 hidden layers, 32 neurons per layer, learning rate 0.001
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Validation Accuracy: 0.8720
Training with 2 hidden layers, 32 neurons per layer, learning rate 0.01
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Validation Accuracy: 0.9000
Training with 2 hidden layers, 64 neurons per layer, learning rate 0.001
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Validation Accuracy: 0.9153
Training 

In [35]:
#Evaluation
# Evaluating the Best Tuned Model
y_best_pred_prob = best_model.predict(X_test)
y_best_pred = np.argmax(y_best_pred_prob, axis=1)

tuned_accuracy = accuracy_score(y_test_actual, y_best_pred)
tuned_precision = precision_score(y_test_actual, y_best_pred, average='weighted')
tuned_recall = recall_score(y_test_actual, y_best_pred, average='weighted')
tuned_f1 = f1_score(y_test_actual, y_best_pred, average='weighted')

# Display tuned model performance
print(f"Tuned Model Performance:")
print(f"Accuracy: {tuned_accuracy:.4f}")
print(f"Precision: {tuned_precision:.4f}")
print(f"Recall: {tuned_recall:.4f}")
print(f"F1 Score: {tuned_f1:.4f}")

# Performance Comparison
diff_accuracy = tuned_accuracy - accuracy
diff_precision = tuned_precision - precision
diff_recall = tuned_recall - recall
diff_f1 = tuned_f1 - f1

print("Performance Improvement After Hyperparameter Tuning:")
print(f"Accuracy Improvement: {diff_accuracy:.4f}")
print(f"Precision Improvement: {diff_precision:.4f}")
print(f"Recall Improvement: {diff_recall:.4f}")
print(f"F1 Score Improvement: {diff_f1:.4f}")


125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Tuned Model Performance:
Accuracy: 0.9233
Precision: 0.9250
Recall: 0.9233
F1 Score: 0.9230
Performance Improvement After Hyperparameter Tuning:
Accuracy Improvement: 0.0085
Precision Improvement: 0.0087
Recall Improvement: 0.0085
F1 Score Improvement: 0.0080
